# Facial Skin Dataset v2 — inspect → review → duplicate-group split → train-ready CSV

**The only real dataset used is** `shanmukhadattaboda/facial-skin-conditions-real-image-collection`, version 2, inside `Face_Skin_dataset`. The folder names are read exactly, including `Hyper_Pigmentation`, `Normal_Skin` and `Skin_cancer`.

**First run:** inventory and hash all originals; emit `review_draft.csv` for source/diagnosis review. **Second run (after review):** read the reviewer-completed CSV, enforce the six-class label gate, create approved-only symlinks (not duplicate image copies), perform the original SHA + pixel hash + pHash + dHash + ResNet + ORB duplicate/group split, and export training-ready `real_all_manifest.csv` and frozen `real_test_manifest.csv`.

**Do not approve photographs only because of folder placement.** A folder is not clinical ground truth. Neither this code nor the Kaggle CC0 license selector verifies the diagnosis, original licensing, or patient-level independence. **No test metrics or model fitting occur in this notebook.** The existing five training notebooks are run separately, with the unchanged real split.

In [ ]:
from pathlib import Path
import json,os,subprocess,sys
# Dataset mount provided by you; the resolver also accepts the ordinary /kaggle/input/<slug>/ layout.
KAGGLE_DATASET_BASE="/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection"
# In Kaggle: attach GitHub repository as input, or enable Internet for a read-only Git clone.
REPO_ROOT=Path("/kaggle/input/EDIT_DERMSYNREAL_REPO/DermSynReal")
CLONE_REPO_IF_NEEDED=True
PINNED_REPO_SHA=""  # Recommended for a final reproducible run.
if not (REPO_ROOT/"src/dermsynreal/real_collection.py").is_file():
    if not CLONE_REPO_IF_NEEDED:
        raise FileNotFoundError("Attach DermSynReal or enable cloning; no alternate data source will be used.")
    REPO_ROOT=Path("/kaggle/working/DermSynReal")
    if not (REPO_ROOT/".git").is_dir():
        subprocess.check_call(["git","clone","--depth","1",
                               "https://github.com/shanmukhdatta/DermSynReal.git",str(REPO_ROOT)])
if PINNED_REPO_SHA:
    if not (REPO_ROOT/".git").is_dir():
        raise ValueError("Pinned SHA requires an actual Git checkout")
    got=subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip()
    assert got==PINNED_REPO_SHA, f"Wrong checkout: {got}"
sys.path.insert(0,str(REPO_ROOT/"src"))
from dermsynreal.real_collection import (
    discover_root,class_directories,inventory_collection,
    stage_approved,export_training_manifests,DATASET_SOURCE
)
root=discover_root(KAGGLE_DATASET_BASE)
print("SOLE REAL DATASET:",root)
print("Exact class mappings:")
for label,path in class_directories(root).items(): print(" ",label,":",path)


## Step 1 — Count, check exact SHA duplicates and prepare review CSV

This step reads the mounted images, hashes their bytes, and flags suspicious filenames (especially potential non-cancer images under Skin Cancer and disorders under Normal). It never changes input photos and never declares a diagnosis. **Exact-duplicate counts cannot be known from the Kaggle screenshots**.

In [ ]:
INVENTORY_OUT="/kaggle/working/real_v2_inventory"
RUN_INVENTORY=True  # First run: True. For reviewed follow-up run, set False.
if RUN_INVENTORY:
    report=inventory_collection(root,INVENTORY_OUT)
    print(json.dumps(report,indent=2))
    print("REVIEW CSV:",Path(INVENTORY_OUT)/"review_draft.csv")
    print("SHA duplicate CSV:",Path(INVENTORY_OUT)/"exact_sha_duplicate_groups.csv")
else:
    print("Skipped inventory. Mount your previously saved inventory.csv in INVENTORY_CSV below.")


## Step 2 — Complete the review outside the notebook

Save the draft and, for every image, verify **original diagnosis, provenance and reviewer**, then choose `approved`, `rejected` or `needs_review`. A flagged mismatch must be rejected/corrected based on source evidence, not overridden blindly. Upload that completed CSV as a private Kaggle input. The pipeline will refuse to split while any item remains `needs_review`. If an image is labeled incorrectly, reject it from this experiment rather than editing its diagnosis without evidence.

In [ ]:
# These two paths refer to SAVED artifacts from the previous inventory/review pass.
# First run: leave RUN_REVIEWED_DUPLICATE_SPLIT=False; just download review_draft.csv.
REVIEWED_CSV="/kaggle/input/EDIT_COMPLETED_REVIEW/reviewed_real_images.csv"
INVENTORY_CSV="/kaggle/input/EDIT_INVENTORY/inventory.csv"
AUDIT_OUT="/kaggle/working/real_v2_label_audit"
STAGING_DIR="/kaggle/working/real_v2_approved_links"
DEDUP_OUT="/kaggle/working/facial_real_split_v2"
RESNET_WEIGHTS_PATH=""  # attach verified ImageNet ResNet50 V2 checkpoint if Kaggle Internet is disabled
RUN_REVIEWED_DUPLICATE_SPLIT=False


## Step 3 — Original-resolution duplicate detection, protective grouping and 70/15/15 splitting

Runs the already-written dedup pipeline with verified pretrained ResNet-50 weights (no random fallback), pHash/dHash, optional ORB and cross-class quarantine. Actual images are copied *once* into the final train/val/test output with original bytes; symlinks only stage the approved subset. A manual review gallery and leakage audit are saved under `reports/`.

In [ ]:
if RUN_REVIEWED_DUPLICATE_SPLIT:
    from dermsynreal.label_gate import write_audit
    reviewed_audit=write_audit(REVIEWED_CSV,AUDIT_OUT,fail_if_blocked=True)
    print("REVIEWED LABEL GATE:",reviewed_audit)
    staged=stage_approved(root,Path(AUDIT_OUT)/"approved_for_split.csv",
        Path(AUDIT_OUT)/"label_audit_summary.json",STAGING_DIR,inventory_csv=INVENTORY_CSV)
    print("APPROVED ONLY:",staged)
    sys.path.insert(0,str(REPO_ROOT/"notebooks/Preprocessing/Real_Data"))
    import pipeline as dedup
    import torch
    assert torch.cuda.is_available(), "Enable Kaggle GPU (T4 recommended) for the ResNet embedding pass"
    settings=dedup.Settings(
        class_dirs={name:str(Path(STAGING_DIR)/name) for name in dedup.CLASS_NAMES},
        output_dir=DEDUP_OUT,resnet_weights_path=RESNET_WEIGHTS_PATH,
        allow_random_weights_for_smoke_test=False,
        seed=42,splits=(0.70,0.15,0.15),orb_verify=True,zip_output=True,
    )
    final_manifest,leakage_report=dedup.run_pipeline(settings)
    if leakage_report["status"]=="FAIL":
        raise RuntimeError("Cross-split hard duplicate evidence; cannot train")
    print("DEDUP/LEAKAGE STATUS:",leakage_report)
else:
    print("Final split locked. Run only after completing reviewer CSV and saving the inventory.")


## Step 4 — Export classifier-ready manifests with stable reviewed image IDs

After split, match every exported photo back to the reviewer-approved source using its SHA-256. The output `real_all_manifest.csv` uses the exact columns expected by the five existing training notebooks. The separate `real_test_manifest.csv` is for the evaluation notebooks. Do not change splits or image IDs between E0, E1, E2, E3, E4 and E5.

In [ ]:
BRIDGED_MANIFESTS="/kaggle/working/real_v2_train_ready"
if RUN_REVIEWED_DUPLICATE_SPLIT:
    result=export_training_manifests(
        Path(DEDUP_OUT)/"reports/split_manifest.csv",
        Path(STAGING_DIR)/"stage_to_original.csv",
        BRIDGED_MANIFESTS,
    )
    print(json.dumps(result,indent=2))
    print("Training CSV:",Path(BRIDGED_MANIFESTS)/"real_all_manifest.csv")
    print("Frozen test CSV:",Path(BRIDGED_MANIFESTS)/"real_test_manifest.csv")
    print("Images root:",Path(DEDUP_OUT))
    print("Zip:",DEDUP_OUT+".zip")
    print("All artifacts must be persisted as private Kaggle outputs/datasets before a notebook session ends.")


## Step 5 — Start the real-only E0 experiments in existing training notebooks

Attach the saved `facial_real_split_v2` folder/ZIP **and** `real_all_manifest.csv` as Kaggle Inputs. Change only the input paths, not the group splits. For each of the five `notebooks/Training/0[1-5]_*.ipynb`:

```python
MANIFEST_CSV = '/kaggle/input/YOUR-SAVED-SPLIT/real_all_manifest.csv'
IMAGE_ROOT = '/kaggle/input/YOUR-SAVED-IMAGES/facial_real_split_v2'
EXPERIMENT = 'real_only'
RUN_FINAL_TEST = False
```

Choose the actual mounted location after saving Kaggle outputs. **Start ResNet50 as a smoke test** (one or two epochs), then EfficientNet-B3, ConvNeXt Tiny, DINOv2 ViT-S/14 and PanDerm Base linear probe. Confirm checkpoint save/reload and held-out real validation macro-F1 before extending training. The real test must remain locked until your models and experiment choices are finalized. E1–E5 need synthetic images generated only from the *approved real training partition*; this uploaded real dataset is the sole real-image source.